# Skin Disease Classification - EfficientNet-B3 Training
Train on Colab GPU, download weights for local use.

**Steps:**
1. Upload your dataset to Google Drive
2. Run all cells
3. Download the trained model

## Step 1: Download Dataset from Kaggle

**Before running:**
1. Go to [kaggle.com/settings](https://www.kaggle.com/settings)
2. Scroll to "API" section → Click "Create New Token"
3. This downloads `kaggle.json` - you'll upload it below

In [ ]:
# Install Kaggle API
!pip install -q kaggle

# Upload your kaggle.json file
from google.colab import files
print("Upload your kaggle.json file:")
uploaded = files.upload()

# Set up Kaggle credentials
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print("✓ Kaggle API configured!")

In [ ]:
# Download the skin disease dataset from Kaggle
# Dataset: https://www.kaggle.com/datasets/ismailpromus/skin-diseases-image-dataset

!kaggle datasets download -d ismailpromus/skin-diseases-image-dataset -p /content/

# Unzip the dataset
!unzip -q /content/skin-diseases-image-dataset.zip -d /content/skin_disease_data
!rm /content/skin-diseases-image-dataset.zip

# Check the structure
!ls /content/skin_disease_data/
print("✓ Dataset downloaded and extracted!")

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install dependencies
!pip install -q tensorflow scikit-learn

In [ ]:
# ============ CONFIGURATION ============
# Dataset location (downloaded from Kaggle above)
DATA_DIR = "/content/skin_disease_data"
OUTPUT_DIR = "/content/model_output"

# Training settings
IMG_SIZE = 300  # EfficientNet-B3 native size
BATCH_SIZE = 32  # Can use larger batch on GPU
EPOCHS_PHASE1 = 15
EPOCHS_PHASE2 = 40
FINE_TUNE_LAYERS = 50

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from tensorflow.keras.applications import EfficientNetB3
from sklearn.utils.class_weight import compute_class_weight
from pathlib import Path
from datetime import datetime

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# Disease classes
CLASS_NAMES = [
    "Eczema",
    "Warts Molluscum and Viral Infections",
    "Melanoma",
    "Atopic Dermatitis",
    "Basal Cell Carcinoma (BCC)",
    "Melanocytic Nevi (NV)",
    "Benign Keratosis-like Lesions (BKL)",
    "Psoriasis and Lichen Planus",
    "Seborrheic Keratoses and Benign Tumors",
    "Tinea Ringworm and Fungal Infections"
]

In [ ]:
# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load dataset
data_path = Path(DATA_DIR) / "IMG_CLASSES"
print(f"Loading dataset from {data_path}")

train_ds = keras.utils.image_dataset_from_directory(
    data_path,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="int"
)

val_ds = keras.utils.image_dataset_from_directory(
    data_path,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="int"
)

class_names = train_ds.class_names
num_classes = len(class_names)
print(f"Found {num_classes} classes: {class_names}")

In [ ]:
# Data augmentation
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomFlip("vertical"),
    layers.RandomRotation(0.3),
    layers.RandomZoom(0.3),
    layers.RandomContrast(0.3),
    layers.RandomBrightness(0.3),
    layers.RandomTranslation(0.1, 0.1),
], name="data_augmentation")

# Preprocessing functions
def preprocess(image, label):
    return tf.cast(image, tf.float32), label

def augment(image, label):
    return data_augmentation(image, training=True), label

# Apply preprocessing
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.map(preprocess, num_parallel_calls=AUTOTUNE)
train_ds = train_ds.map(augment, num_parallel_calls=AUTOTUNE)
train_ds = train_ds.prefetch(AUTOTUNE)

val_ds = val_ds.map(preprocess, num_parallel_calls=AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

In [ ]:
# Compute class weights
labels = []
for _, label_batch in train_ds:
    labels.extend(label_batch.numpy())
labels = np.array(labels)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)
class_weight_dict = {i: w for i, w in enumerate(class_weights)}
print(f"Class weights: {class_weight_dict}")

In [ ]:
# Create EfficientNet-B3 model
base_model = EfficientNetB3(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    pooling=None
)
base_model.trainable = False  # Freeze for Phase 1

# Build model
inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D(name="global_avg_pool")(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(256, activation="relu", kernel_regularizer=keras.regularizers.l2(0.01))(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)

model = keras.Model(inputs, outputs, name="skin_disease_classifier")
model.summary()

In [ ]:
# Callbacks
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

callbacks_list = [
    callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
    callbacks.ModelCheckpoint(
        f"{OUTPUT_DIR}/checkpoint_{timestamp}.keras",
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    )
]

In [ ]:
# ===== PHASE 1: Train custom head =====
print("="*50)
print("PHASE 1: Training custom classification head")
print("="*50)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE1,
    class_weight=class_weight_dict,
    callbacks=callbacks_list
)

In [ ]:
# ===== PHASE 2: Fine-tune base model =====
print("="*50)
print(f"PHASE 2: Fine-tuning top {FINE_TUNE_LAYERS} layers")
print("="*50)

# Unfreeze top layers
base_model.trainable = True
for layer in base_model.layers[:-FINE_TUNE_LAYERS]:
    layer.trainable = False

# Recompile with lower learning rate
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Update checkpoint path
callbacks_list[2] = callbacks.ModelCheckpoint(
    f"{OUTPUT_DIR}/checkpoint_finetune_{timestamp}.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE2,
    class_weight=class_weight_dict,
    callbacks=callbacks_list,
    initial_epoch=len(history1.history['loss'])
)

In [ ]:
# Save final model
model.save(f"{OUTPUT_DIR}/efficientnetb3_skin_disease.keras")
print(f"Model saved to {OUTPUT_DIR}/efficientnetb3_skin_disease.keras")

# Save class names
with open(f"{OUTPUT_DIR}/class_names.txt", "w") as f:
    for i, name in enumerate(class_names):
        f.write(f"{i}: {name}\n")
print("Class names saved")

In [ ]:
# Final evaluation
print("="*50)
print("Final Evaluation")
print("="*50)
val_loss, val_accuracy = model.evaluate(val_ds)
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")

In [ ]:
# Download the model
from google.colab import files

# Download trained model
files.download(f"{OUTPUT_DIR}/efficientnetb3_skin_disease.keras")
files.download(f"{OUTPUT_DIR}/class_names.txt")

In [ ]:
# Alternative: Copy to Google Drive
!cp {OUTPUT_DIR}/efficientnetb3_skin_disease.keras /content/drive/MyDrive/
!cp {OUTPUT_DIR}/class_names.txt /content/drive/MyDrive/
print("Model copied to Google Drive root!")